# Lesson 16: Recurrent Neural Networks (RNNs), LSTMs & GRUs

In this notebook, you will:
1. Build a PyTorch **Bidirectional LSTM** sequence classifier with `nn.Embedding` and `nn.LSTM`.
2. Handle variable-length sequences cleanly using padding.

In [ ]:
import torch
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hid_dim=64, num_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, num_classes)

    def forward(self, x):
        e = self.emb(x)
        _, (h_n, _) = self.lstm(e)
        # Concatenate final forward and backward hidden states
        feat = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.fc(feat)

model = BiLSTMClassifier(vocab_size=100)
batch = torch.randint(1, 100, (8, 12))
logits = model(batch)
print("BiLSTM logits shape:", tuple(logits.shape))

## Part 2: Comparing GRU vs LSTM Parameters

In [ ]:
gru = nn.GRU(32, 64, batch_first=True, bidirectional=True)
lstm_params = sum(p.numel() for p in model.lstm.parameters())
gru_params = sum(p.numel() for p in gru.parameters())
print(f"BiLSTM params: {lstm_params:,} | BiGRU params: {gru_params:,}")